In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [22]:
def preprocess_input(new_input, feature_columns):
    new_input = new_input.copy()

    # Remove columns not used during training
    columns_to_drop = [
        'Student_ID',
        'Job_ID',
        'Gender',
        'Age',
        'Location',
        'Job_Match'
    ]

    new_input = new_input.drop(
        columns=columns_to_drop,
        errors='ignore'
    )

    # Encode categorical columns if they are still present
    categorical_columns = [
        col for col in ['Job_Title', 'Degree_Program']
        if col in new_input.columns
    ]

    new_input = pd.get_dummies(
        new_input,
        columns=categorical_columns,
        dtype=int
    )

    # Keep only the columns used during model training
    new_input = new_input.reindex(
        columns=feature_columns,
        fill_value=0
    )

    return new_input

In [23]:
def predict_job_match(new_input, model, feature_columns):
    processed_input = preprocess_input(
        new_input,
        feature_columns
    )

    # Predict the class
    prediction = model.predict(processed_input)[0]

    # Get estimated probability for class 1
    match_class_index = list(model.classes_).index(1)

    probability = model.predict_proba(processed_input)[
        0, match_class_index
    ]

    match_percentage = round(float(probability * 100), 2)

    if prediction == 1:
        result = "Job Match"
    else:
        result = "Job Not Match"

    return {
        "result": result,
        "prediction": int(prediction),
        "match_percentage": match_percentage
    }

In [24]:
import pickle

with open('../static/model/model.pickle', 'rb') as file:
    model = pickle.load(file)

with open('../static/model/feature_columns.pickle', 'rb') as file:
    feature_columns = pickle.load(file)

In [25]:
new_input = pd.DataFrame([{
    'Academic_Performance': 3.7,
    'Certifications_Count': 2,
    'Internship_Experience': 6,
    'Skill_Programming': 7,
    'Skill_Database_SQL': 7,
    'Skill_Version_Control_Git': 7,
    'Skill_Problem_Solving': 8,
    'Skill_Communication': 7,
    'Required_Programming': 8,
    'Required_Database_SQL': 8,
    'Required_Version_Control_Git': 8,
    'Required_Problem_Solving': 8,
    'Required_Communication': 8,
    'Min_Experience_Months': 6,
    'Job_Title': 'AI Engineer',
    'Degree_Program': 'AI Engineering'
}])

result = predict_job_match(new_input, model, feature_columns)

print(result)

{'result': 'Job Match', 'prediction': 1, 'match_percentage': 82.02}
